<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **ranking / scoring** task. The goal is not to predict a single yes/no label, but to assign a priority score to each page so an editor can review the most promising refresh candidates first. A ranking framing fits because the business action is an ordered queue of pages, and the main success metric is how many of the highest-ranked pages are truly useful review candidates.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Example columns:", list(df.columns[:8]))

Rows: 30000
Columns: 44
Example columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent']


## 2. Target or proxy

The target is a **declining-label outcome** for a page, derived from an observed later signal in the dataset. In this repo, that is represented by the label that indicates whether the page is declining. This is an observed outcome rather than a hand-written rule, which makes it appropriate for a supervised task.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["is_declining_label"].value_counts().to_dict())
print("Base rate:", round(df["is_declining_label"].mean(), 3))

{1: 16262, 0: 13738}
Base rate: 0.542


## 3. Success metric

The clearest metric is **Precision@50**. It measures how many of the top 50 ranked pages are truly useful refresh candidates. A higher Precision@50 is better because the editor's time is limited and the first pages reviewed matter most.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"Base rate: {base_rate:.3f}")
print("Precision@50 is meaningful here because the top-ranked queue is the action surface.")

Base rate: 0.542
Precision@50 is meaningful here because the top-ranked queue is the action surface.


## 4. The unit of analysis, as a real dataframe

One row is one **page**. The dataframe captures a page-level observation with signals such as impressions, position, CTR, and content age, plus the declining-label outcome used for evaluation.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(df.head(3).to_string())

             content_id          client_id  search_volume  competition competition_level   cpc     content_type    main_intent  word_count  char_count provider_used              model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier   ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
0  content_304f48230142  client_f369cb89fc           10.0         0.67              HIGH  2.05  keyword article  transactional      3221.0     20457.0           NaN        gemini-2.5-flash             3803          29             22            17         16                     1         

## 5. Why ML beats a fixed rule here

A fixed rule is too brittle because the useful signal is spread across many interacting features, such as visibility, engagement, content age, and page context. The relationship is not simple enough to capture with one if-statement, but it is still structured enough that a model can learn a ranking function from the data.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

signal_cols = ["days_with_impressions", "avg_position", "ctr", "word_count", "char_count"]
print(df[signal_cols].describe().round(2).to_string())

       days_with_impressions  avg_position       ctr  word_count  char_count
count               30000.00      30000.00  30000.00    22301.00    22301.00
mean                   61.45         16.34      0.51     3107.76    20665.28
std                    32.69         15.22      3.28     1452.38    10115.34
min                     1.00          0.00      0.00        8.00       40.00
25%                    31.00          6.20      0.00     2413.00    15644.00
50%                    81.00         10.80      0.07     2877.00    19116.00
75%                    88.00         22.30      0.29     3666.00    24011.00
max                    88.00        245.00    100.00     9546.00   111158.00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.